In [1]:
import numpy as np
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import multiprocessing as mp
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy import stats
import os
from stoch_sim_model import *
from optimize_nets import *
import seaborn as sns

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'

In [2]:
num_pnts = 21
sample_virs = np.array(np.meshgrid(d_S*np.logspace(np.log10(1/sim_duration/d_S), 2.0, num_pnts), # vary d_I
                                   K_IE*np.logspace(0.0, 3.0, num_pnts), # vary K_IE
                                   b_I*np.logspace(-1, 1, num_pnts) # vary b_I
                                           )).T.reshape(-1,3)

default_reg = act_psis + NM_psis + EM_psis + exp_psis

d="/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/"

In [ ]:
# run simulations for extreme scenarios
run(batch = [-1], outdir=d, inf_sample = sample_virs, comment="single-reg-vary-virs", vir_model = "indep_harm", default_reg = default_reg, num_cpu = 40)

Running 9261 simulations in batch #-1


In [ ]:
d_file = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/sim_batch_-1-1-prim-single-reg-vary-virs.pkl'
with open(d_file, 'rb') as f:   
    data_dict = pickle.load(f)

n = len(sample_virs)
parameters = data_dict["parameters"]
sim_summary = data_dict["summary_stats"]
# dyn = data_dict["cell_time_series"]
# p_diff = data_dict["prim_diff_bias"]

In [ ]:
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
data_df = pd.DataFrame(np.hstack((parameters, sim_summary)), columns = [i for i in var_names]).dropna(thresh=1)
data_df['T_min_pI_corrected'] = 20*(data_df['T_min_pI'] == 0) + data_df['T_min_pI']

with pd.option_context('display.max_columns', None):
    display(data_df)

In [ ]:
# Plot simulated immune dynamics
fig, (ax1, ax2, ax3) = plt.subplots(3, dpi = 150, constrained_layout=True, sharex = False, figsize=(6,9))


ax1.plot(data_df.groupby(['K_IE'], as_index=False).mean()['K_IE']/np.max(data_df['K_IE']), data_df.groupby(['K_IE'], as_index=False).mean()['min_pS']/S_0, 'o--', label = 'Antigenicity, $K_{I,E}$')
ax1.plot(data_df.groupby(['d_I'], as_index=False).mean()['d_I']/np.max(data_df['d_I']), data_df.groupby(['d_I'], as_index=False).mean()['min_pS']/S_0, 'o--', label = 'Mortality, $d_{I}$')
ax1.plot(data_df.groupby(['b_I'], as_index=False).mean()['b_I']/np.max(data_df['b_I']), data_df.groupby(['b_I'], as_index=False).mean()['min_pS']/S_0, 'o--', label = 'Fecundity, $b_{I}$')
ax1.axvline(x = 1/sim_duration/np.max(data_df['d_I']), color = 'k', linestyle = '--', label = "Lower bound \n for Mortality")
ax1.axvline(x = 1/(sim_duration*S_0)*(1/np.max(data_df['b_I'])), color = 'r', linestyle = '--', label = "Lower bound \n for fecundity")
ax1.semilogx()
ax1.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')
ax1.set(ylabel = 'Minumum \n suceptible fraction')

ax2.plot(data_df.groupby(['K_IE'], as_index=False).mean()['K_IE']/np.max(data_df['K_IE']), data_df.groupby(['K_IE'], as_index=False).mean()['p_load']/S_0, 'o--', label = 'Antigenicity, $K_{I,E}$')
ax2.plot(data_df.groupby(['d_I'], as_index=False).mean()['d_I']/np.max(data_df['d_I']), data_df.groupby(['d_I'], as_index=False).mean()['p_load']/S_0, 'o--', label = 'Mortality, $d_{I}$')
ax2.plot(data_df.groupby(['b_I'], as_index=False).mean()['b_I']/np.max(data_df['b_I']), data_df.groupby(['b_I'], as_index=False).mean()['p_load']/S_0, 'o--', label = 'Fecundity, $b_{I}$')
ax2.axvline(x = 1/sim_duration/np.max(data_df['d_I']), color = 'k', linestyle = '--', label = "Lower bound \n for Mortality")
ax2.axvline(x = 1/(sim_duration*S_0)*(1/np.max(data_df['b_I'])), color = 'r', linestyle = '--', label = "Lower bound \n for fecundity")
ax2.semilogx()
ax2.set(ylabel = 'Cumulative pathogen load')
ax2.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')

ax3.plot(data_df.groupby(['K_IE'], as_index=False).mean()['K_IE']/np.max(data_df['K_IE']), data_df.groupby(['K_IE'], as_index=False).mean()['T_min_pI_corrected'], 'o--', label = 'Antigenicity, $K_{I,E}$')
ax3.plot(data_df.groupby(['d_I'], as_index=False).mean()['d_I']/np.max(data_df['d_I']), data_df.groupby(['d_I'], as_index=False).mean()['T_min_pI_corrected'], 'o--', label = 'Mortality, $d_{I}$')
ax3.plot(data_df.groupby(['b_I'], as_index=False).mean()['b_I']/np.max(data_df['b_I']), data_df.groupby(['b_I'], as_index=False).mean()['T_min_pI_corrected'], 'o--', label = 'Fecundity, $b_{I}$')
ax3.axvline(x = 1/sim_duration/np.max(data_df['d_I']), color = 'k', linestyle = '--', label = "Lower bound \n for Mortality")
ax3.axvline(x = 1/(sim_duration*S_0)*(1/np.max(data_df['b_I'])), color = 'r', linestyle = '--', label = "Lower bound \n for fecundity")
ax3.set(ylabel = 'Time for clearance', xlabel = 'Max-scaled parameter')
ax3.semilogx()
ax3.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')


plt.savefig('_figs/vir_parameter_lineplots', dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3), sharex=False, sharey=False)

axs[0].tricontour(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], 
                  data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], 
                  data_df.groupby(['K_IE','d_I'], as_index=False).mean()['min_pS']/S_0, linewidths=0.1, colors='k')
im0 = axs[0].tricontourf(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], 
                        data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], 
                        data_df.groupby(['K_IE','d_I'], as_index=False).mean()['min_pS']/S_0, cmap="coolwarm_r")

cb = fig.colorbar(im0, ax=axs[0], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

axs[1].tricontour(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], 
                  data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], 
                  data_df.groupby(['K_IE','b_I'], as_index=False).mean()['min_pS']/S_0, linewidths=0.1, colors='k')
im1 = axs[1].tricontourf(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], 
                        data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], 
                        data_df.groupby(['K_IE','b_I'], as_index=False).mean()['min_pS']/S_0, cmap="coolwarm_r")

cb = fig.colorbar(im1, ax=axs[1], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

axs[2].tricontour(data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], 
                  data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], 
                  data_df.groupby(['b_I','d_I'], as_index=False).mean()['min_pS']/S_0, linewidths=0.1, colors='k')
im2 = axs[2].tricontourf(data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], 
                        data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], 
                        data_df.groupby(['b_I','d_I'], as_index=False).mean()['min_pS']/S_0, cmap="coolwarm_r")

cb = fig.colorbar(im2, ax=axs[2], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

axs[0].set_title(r"Antigenicity $K_{I,E}$ vs. Mortality $d_I$", fontsize = 12)
axs[1].set_title(r"Antigenicity $K_{I,E}$ vs. Fecundity $b_I$", fontsize = 12)
axs[2].set_title(r"Fecundity $b_I$ vs. Mortality $d_I$", fontsize = 12)

axs[0].set_ylabel("Mortality, $d_I$", fontsize = 10)
axs[0].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[1].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[1].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[2].set_ylabel("Mortality, $d_I$", fontsize = 10)
axs[2].set_xlabel("Fecundity, $b_I$", fontsize = 10)

for j, ax in enumerate(axs.flat):
    ax.semilogy()
    ax.semilogx()

fig.subplots_adjust(wspace=.5, hspace=.5)
    
plt.savefig('_figs/vir_parameter_w_Smin_contour', dpi=300, bbox_inches='tight')

In [ ]:
# fig, axs = plt.subplots(1, 3, figsize=(15, 3), sharex=False, sharey=False)

# axs[0].tricontour(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], 
#                   data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], 
#                   data_df.groupby(['K_IE','d_I'], as_index=False).mean()['p_load']/S_0, linewidths=0.2, colors='k')
# im0 = axs[0].tricontourf(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], 
#                         data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], 
#                         data_df.groupby(['K_IE','d_I'], as_index=False).mean()['p_load']/S_0, cmap="coolwarm")

# cb = fig.colorbar(im0, ax=axs[0], orientation='vertical')
# cb.set_label("Cumulative pathogen load", fontsize = 12)

# axs[1].tricontour(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], 
#                   data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], 
#                   data_df.groupby(['K_IE','b_I'], as_index=False).mean()['p_load']/S_0, linewidths=0.2, colors='k')
# im1 = axs[1].tricontourf(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], 
#                         data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], 
#                         data_df.groupby(['K_IE','b_I'], as_index=False).mean()['p_load']/S_0, cmap="coolwarm")

# cb = fig.colorbar(im1, ax=axs[1], orientation='vertical')
# cb.set_label("Cumulative pathogen load", fontsize = 12)

# axs[2].tricontour(data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], 
#                   data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], 
#                   data_df.groupby(['b_I','d_I'], as_index=False).mean()['p_load']/S_0, linewidths=0.2, colors='k')
# im2 = axs[2].tricontourf(data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], 
#                         data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], 
#                         data_df.groupby(['b_I','d_I'], as_index=False).mean()['p_load']/S_0, cmap="coolwarm")

# cb = fig.colorbar(im2, ax=axs[2], orientation='vertical')
# cb.set_label("Cumulative pathogen load", fontsize = 12)

# axs[0].set_title(r"Antigenicity $K_{I,E}$ vs. Mortality $d_I$", fontsize = 12)
# axs[1].set_title(r"Antigenicity $K_{I,E}$ vs. Fecundity $b_I$", fontsize = 12)
# axs[2].set_title(r"Fecundity $b_I$ vs. Mortality $d_I$", fontsize = 12)

# axs[0].set_ylabel("Mortality, $d_I$", fontsize = 10)
# axs[0].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
# axs[1].set_ylabel("Fecundity, $b_I$", fontsize = 10)
# axs[1].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
# axs[2].set_ylabel("Mortality, $d_I$", fontsize = 10)
# axs[2].set_xlabel("Fecundity, $b_I$", fontsize = 10)

# for j, ax in enumerate(axs.flat):
#     ax.semilogy()
#     ax.semilogx()

# fig.subplots_adjust(wspace=.5, hspace=.5)
    
# plt.savefig('_figs/vir_parameter_w_pload_contour', dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3), sharex=False, sharey=False)

im0 = axs[0].scatter(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], marker = 's', s = 150,
                    c = data_df.groupby(['K_IE','d_I'], as_index=False).mean()['min_pS']/S_0, cmap='coolwarm_r', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im0, ax=axs[0], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

im1 = axs[1].scatter(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], marker = 's', s = 150,
               c = data_df.groupby(['K_IE','b_I'], as_index=False).mean()['min_pS']/S_0, cmap='coolwarm_r', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im1, ax=axs[1], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

im2 = axs[2].scatter(data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], marker = 's', s = 150,
               c = data_df.groupby(['b_I','d_I'], as_index=False).mean()['min_pS']/S_0, cmap='coolwarm_r', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im2, ax=axs[2], orientation='vertical')
cb.set_label("Minumum \n suceptible fraction", fontsize = 12)

axs[0].set_title(r"Antigenicity $K_{I,E}$ vs. Mortality $d_I$", fontsize = 12)
axs[1].set_title(r"Antigenicity $K_{I,E}$ vs. Fecundity $b_I$", fontsize = 12)
axs[2].set_title(r"Mortality $d_I$ vs. Fecundity $b_I$", fontsize = 12)

axs[0].set_ylabel("Mortality, $d_I$", fontsize = 10)
axs[0].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[1].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[1].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[2].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[2].set_xlabel("Mortality, $d_I$", fontsize = 10)


for j, ax in enumerate(axs.flat):
    ax.semilogy()
    ax.semilogx()

fig.subplots_adjust(wspace=.5, hspace=.5)
    
plt.savefig('_figs/vir_parameter_w_Smin_heatmaps', dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3), sharex=False, sharey=False)

im0 = axs[0].scatter(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], marker = 's', s = 150,
                    c = data_df.groupby(['K_IE','d_I'], as_index=False).mean()['p_load'], cmap='coolwarm', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im0, ax=axs[0], orientation='vertical')
cb.set_label("Cumulative pathogen load", fontsize = 12)

im1 = axs[1].scatter(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], marker = 's', s = 150,
               c = data_df.groupby(['K_IE','b_I'], as_index=False).mean()['p_load'], cmap='coolwarm', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im1, ax=axs[1], orientation='vertical')
cb.set_label("Cumulative pathogen load", fontsize = 12)

im2 = axs[2].scatter(data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], marker = 's', s = 150,
               c = data_df.groupby(['b_I','d_I'], as_index=False).mean()['p_load'], cmap='coolwarm', norm=mpl.colors.LogNorm())
cb = fig.colorbar(im2, ax=axs[2], orientation='vertical')
cb.set_label("Cumulative pathogen load", fontsize = 12)

axs[0].set_title(r"Antigenicity $K_{I,E}$ vs. Mortality $d_I$", fontsize = 12)
axs[1].set_title(r"Antigenicity $K_{I,E}$ vs. Fecundity $b_I$", fontsize = 12)
axs[2].set_title(r"Mortality $d_I$ vs. Fecundity $b_I$", fontsize = 12)

axs[0].set_ylabel("Mortality, $d_I$", fontsize = 10)
axs[0].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[1].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[1].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[2].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[2].set_xlabel("Mortality, $d_I$", fontsize = 10)

for j, ax in enumerate(axs.flat):
    ax.semilogy()
    ax.semilogx()

fig.subplots_adjust(wspace=.5, hspace=.5)
    
plt.savefig('_figs/vir_parameter_w_pload_heatmaps', dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3), sharex=False, sharey=False)

im0 = axs[0].scatter(data_df.groupby(['K_IE','d_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','d_I'], as_index=False).mean()['d_I'], marker = 's', s = 150,
                    c = data_df.groupby(['K_IE','d_I'], as_index=False).mean()['T_min_pI_corrected'], cmap='coolwarm')
cb = fig.colorbar(im0, ax=axs[0], orientation='vertical')
cb.set_label("Time for clearance", fontsize = 12)

im1 = axs[1].scatter(data_df.groupby(['K_IE','b_I'], as_index=False).mean()['K_IE'], data_df.groupby(['K_IE','b_I'], as_index=False).mean()['b_I'], marker = 's', s = 110,
               c = data_df.groupby(['K_IE','b_I'], as_index=False).mean()['T_min_pI_corrected'], cmap='coolwarm')
cb = fig.colorbar(im1, ax=axs[1], orientation='vertical')
cb.set_label("Time for clearance", fontsize = 12)

im2 = axs[2].scatter(data_df.groupby(['b_I','d_I'], as_index=False).mean()['d_I'], data_df.groupby(['b_I','d_I'], as_index=False).mean()['b_I'], marker = 's', s = 110,
               c = data_df.groupby(['b_I','d_I'], as_index=False).mean()['T_min_pI_corrected'], cmap='coolwarm')
cb = fig.colorbar(im2, ax=axs[2], orientation='vertical')
cb.set_label("Time for clearance", fontsize = 12)

axs[0].set_title(r"Antigenicity $K_{I,E}$ vs. Mortality $d_I$", fontsize = 12)
axs[1].set_title(r"Antigenicity $K_{I,E}$ vs. Fecundity $b_I$", fontsize = 12)
axs[2].set_title(r"Mortality $d_I$ vs. Fecundity $b_I$", fontsize = 12)

axs[0].set_ylabel("Mortality, $d_I$", fontsize = 10)
axs[0].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[1].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[1].set_xlabel("Antigenicity, $K_{I,E}$", fontsize = 10)
axs[2].set_ylabel("Fecundity, $b_I$", fontsize = 10)
axs[2].set_xlabel("Mortality, $d_I$", fontsize = 10)

for j, ax in enumerate(axs.flat):
    ax.semilogy()
    ax.semilogx()

fig.subplots_adjust(wspace=.5, hspace=.5)
    
plt.savefig('_figs/vir_parameter_w_ptiming_heatmaps', dpi=300, bbox_inches='tight')